# Three levels of improvement, with no model training

This notebook distinguishes **changing an answer**, **changing the agent**,
and **changing the procedure that improves the agent**.

Our agent calculates invoices. Its fixed Python interpreter is already able
to apply rules, but its persistent configuration initially omits quantity,
discounts, and shipping. The agent proposes **data edits**, not Python code.
This is a constructive counterexample to “a non-coding agent cannot improve.”

**Scope:** this is a deterministic, finite search demonstration, not an LLM
experiment. Candidate edits and the interpreter are human-authored. A real
LLM could propose the same structured patches from execution feedback, but
that does not guarantee the resulting changes are useful. The evaluator must
test them. No neural model, API key, package installation, or network is used.

Run the cells from top to bottom. Every code cell uses Python's standard library.


In [1]:
from dataclasses import dataclass, replace, asdict
from decimal import Decimal
from pathlib import Path
import json
from typing import Callable


@dataclass(frozen=True)
class Invoice:
    unit_price: Decimal
    quantity: int = 1
    discount: Decimal = Decimal("0")
    shipping: Decimal = Decimal("0")


def invoice(price, quantity=1, discount="0", shipping="0"):
    return Invoice(Decimal(str(price)), quantity,
                   Decimal(str(discount)), Decimal(str(shipping)))


@dataclass(frozen=True)
class AgentRules:
    use_quantity: bool = False
    use_discount: bool = False
    use_shipping: bool = False


@dataclass(frozen=True)
class ImprovementPolicy:
    edit_mode: str = "single"  # single: one field per candidate; bundle: all fields


def answer(task: Invoice, rules: AgentRules) -> Decimal:
    subtotal = task.unit_price * (task.quantity if rules.use_quantity else 1)
    if rules.use_discount:
        subtotal *= 1 - task.discount
    if rules.use_shipping:
        subtotal += task.shipping
    return subtotal.quantize(Decimal("0.01"))


def reference_answer(task: Invoice) -> Decimal:
    # Independent, fixed business specification. Shipping is not discounted.
    total = task.unit_price * task.quantity * (1 - task.discount) + task.shipping
    return total.quantize(Decimal("0.01"))


def accuracy(rules: AgentRules, tasks: list[Invoice]) -> float:
    return sum(answer(t, rules) == reference_answer(t) for t in tasks) / len(tasks)


print("Initial persistent rules:", asdict(AgentRules()))


Initial persistent rules: {'use_quantity': False, 'use_discount': False, 'use_shipping': False}


## 1. Output refinement: fix this answer, forget the fix

The verifier provides specific feedback about a discounted invoice. The agent
retries using temporary rules. Nothing is retained for the next request.
The verifier has an exact oracle in this toy; most real tasks do not.


In [2]:
initial_agent = AgentRules()
task = invoice("100", discount="0.20")
first_answer = answer(task, initial_agent)
feedback = "This invoice has a 20% discount; apply it to the subtotal."
temporary_rules = replace(initial_agent, use_discount=True)
refined_answer = answer(task, temporary_rules)
next_request_answer = answer(task, initial_agent)

print("First answer:          $", first_answer, sep="")
print("Verifier feedback:", feedback)
print("Refined answer:        $", refined_answer, sep="")
print("Next request:          $", next_request_answer, sep="")
print("Persistent rules:", asdict(initial_agent))
assert refined_answer == reference_answer(task) == Decimal("80.00")
assert next_request_answer == first_answer == Decimal("100.00")
assert initial_agent == AgentRules()


First answer:          $100.00
Verifier feedback: This invoice has a 20% discount; apply it to the subtotal.
Refined answer:        $80.00
Next request:          $100.00
Persistent rules: {'use_quantity': False, 'use_discount': False, 'use_shipping': False}


## A shared improvement engine

`evaluate` returns a score, trace details, and proposed structured patches.
The tiny diagnosis function knows the invoice specification. This deliberately
makes failure diagnosis easy so we can isolate persistence and recursion.

`propose` interprets an editable improvement policy:

- `single`: generate one candidate for each suggested field change.
- `bundle`: generate one candidate containing all suggested field changes.

`improve` tests candidates and retains a strictly better state. The same
engine works on **AgentRules** and on **ImprovementPolicy**. Its search budget
counts candidate evaluations; the baseline is evaluated once separately.
With one candidate evaluation, the order of single edits matters. We keep the
order fixed and make it visible, rather than claiming an optimal search.


In [3]:
@dataclass
class Evaluation:
    score: float
    patches: list[tuple[str, object]]
    details: dict


def evaluate_agent(rules: AgentRules, tasks: list[Invoice]) -> Evaluation:
    traces = []
    repairs = set()
    for task in tasks:
        actual, expected = answer(task, rules), reference_answer(task)
        traces.append({"actual": str(actual), "expected": str(expected),
                       "passed": actual == expected})
        if actual != expected:
            if task.quantity != 1 and not rules.use_quantity:
                repairs.add("use_quantity")
            if task.discount != 0 and not rules.use_discount:
                repairs.add("use_discount")
            if task.shipping != 0 and not rules.use_shipping:
                repairs.add("use_shipping")
    # Human-authored patch vocabulary: this is not open-ended code synthesis.
    order = ("use_quantity", "use_discount", "use_shipping")
    patches = [(key, True) for key in order if key in repairs]
    return Evaluation(accuracy(rules, tasks), patches, {"traces": traces})


def propose(state, feedback: Evaluation, policy: ImprovementPolicy):
    if policy.edit_mode == "single":
        return [replace(state, **{key: value}) for key, value in feedback.patches]
    if policy.edit_mode == "bundle":
        return [replace(state, **dict(feedback.patches))] if feedback.patches else []
    raise ValueError(f"Unknown edit mode: {policy.edit_mode}")


def improve(state, evaluate: Callable, policy: ImprovementPolicy, budget: int):
    if budget < 0:
        raise ValueError("Budget must be nonnegative")
    best = state
    best_feedback = evaluate(best)
    history = [{"kind": "baseline", "state": asdict(best),
                "score": best_feedback.score}]
    evaluations = 0
    while evaluations < budget:
        candidates = propose(best, best_feedback, policy)
        if not candidates:
            break
        accepted_in_round = False
        # Proposals come from the current best state's execution feedback.
        for candidate in candidates:
            if evaluations >= budget:
                break
            candidate_feedback = evaluate(candidate)
            evaluations += 1
            accepted = candidate_feedback.score > best_feedback.score
            history.append({"kind": "candidate", "state": asdict(candidate),
                            "score": candidate_feedback.score, "accepted": accepted})
            if accepted:
                best, best_feedback = candidate, candidate_feedback
                accepted_in_round = True
                break  # Re-diagnose after each accepted edit.
        if not accepted_in_round:
            break
    return best, {"candidate_evaluations": evaluations, "history": history}


development_tasks = [
    invoice("10", quantity=3),
    invoice("100", discount="0.20"),
    invoice("20", shipping="5"),
]
# These tasks are never passed to the ordinary agent improvement loop.
held_out_tasks = [
    invoice("7.50", quantity=4),
    invoice("60", discount="0.25"),
    invoice("13", shipping="2.75"),
    invoice("12.50", quantity=3, discount="0.10", shipping="4"),
]
initial_policy = ImprovementPolicy("single")
agent_evaluator = lambda rules: evaluate_agent(rules, development_tasks)
print("Candidate patches from failures:", agent_evaluator(initial_agent).patches)


Candidate patches from failures: [('use_quantity', True), ('use_discount', True), ('use_shipping', True)]


## 2. Agent self-improvement: retain the new rule

The fixed improvement policy proposes changes to persistent agent rules.
We allow **one** candidate evaluation. It learns quantity because that is
first in the explicit patch order. Reloading the saved rule changes future
behavior on new invoices. The improvement policy stays unchanged.

JSON files are written under `toy_artifacts/` in the current working directory.


In [4]:
learned_agent, agent_log = improve(initial_agent, agent_evaluator,
                                 initial_policy, budget=1)
artifacts = Path("toy_artifacts")
artifacts.mkdir(exist_ok=True)
(artifacts / "agent_rules.json").write_text(json.dumps(asdict(learned_agent), indent=2))
reloaded_agent = AgentRules(**json.loads((artifacts / "agent_rules.json").read_text()))

new_quantity_task = invoice("7.50", quantity=4)
print("Retained rules:", asdict(reloaded_agent))
print("New invoice before: $", answer(new_quantity_task, initial_agent), sep="")
print("New invoice after:  $", answer(new_quantity_task, reloaded_agent), sep="")
print("Development accuracy:", accuracy(reloaded_agent, development_tasks))
print("Held-out accuracy:", accuracy(reloaded_agent, held_out_tasks))
print("Improvement policy remains:", asdict(initial_policy))
assert answer(new_quantity_task, reloaded_agent) == Decimal("30.00")
assert accuracy(reloaded_agent, held_out_tasks) == 0.25
assert agent_log["candidate_evaluations"] == 1


Retained rules: {'use_quantity': True, 'use_discount': False, 'use_shipping': False}
New invoice before: $7.50
New invoice after:  $30.00
Development accuracy: 0.3333333333333333
Held-out accuracy: 0.25
Improvement policy remains: {'edit_mode': 'single'}


## 3. Recursive self-improvement: retain a better improvement policy

Now the object being optimized is `ImprovementPolicy`, not `AgentRules`.
To evaluate a candidate policy, run fresh agent-learning episodes with that
policy. Each episode has **the same one-candidate adaptation budget**. Score
the resulting agents on episode validation tasks with different invoice values.

The meta-level asks: “Does this policy produce better agent improvements?”
Feedback proposes `bundle` if the one-edit policy leaves the demonstrated
failures unresolved. That alternative is human-authored, not invented.
The **same `improve` function** applies the existing policy to improve itself.
The retained policy then controls future improvement runs, including a later
meta-level run. The recursive boundary is the editable proposal mechanism;
the acceptance function, interpreter, and external oracle remain fixed.


In [5]:
meta_development_episodes = [
    ([invoice("8", quantity=2), invoice("50", discount="0.10"),
      invoice("15", shipping="3")],
     [invoice("11", quantity=5), invoice("90", discount="0.30"),
      invoice("17", shipping="6")]),
    ([invoice("9", quantity=4, discount="0.20", shipping="2")],
     [invoice("14", quantity=2, discount="0.15", shipping="5")]),
]


def evaluate_policy(policy: ImprovementPolicy, episodes) -> Evaluation:
    scores, logs = [], []
    for training, validation in episodes:
        fresh_agent = AgentRules()
        adapted, log = improve(fresh_agent,
                               lambda rules: evaluate_agent(rules, training),
                               policy, budget=1)
        scores.append(accuracy(adapted, validation))
        logs.append(log)
    score = sum(scores) / len(scores)
    patches = [("edit_mode", "bundle")] if score < 1 and policy.edit_mode == "single" else []
    return Evaluation(score, patches, {"episode_scores": scores, "adaptation_logs": logs})


meta_evaluator = lambda policy: evaluate_policy(policy, meta_development_episodes)
learned_policy, meta_log = improve(initial_policy, meta_evaluator,
                                  policy=initial_policy, budget=1)
(artifacts / "improvement_policy.json").write_text(
    json.dumps(asdict(learned_policy), indent=2))
reloaded_policy = ImprovementPolicy(**json.loads(
    (artifacts / "improvement_policy.json").read_text()))
print("Policy before:", asdict(initial_policy))
print("Policy after: ", asdict(reloaded_policy))
print("Meta-validation score before:", meta_evaluator(initial_policy).score)
print("Meta-validation score after: ", meta_evaluator(reloaded_policy).score)
print("Meta search candidate evaluations:", meta_log["candidate_evaluations"])
assert reloaded_policy.edit_mode == "bundle"
assert meta_evaluator(reloaded_policy).score == 1.0


Policy before: {'edit_mode': 'single'}
Policy after:  {'edit_mode': 'bundle'}
Meta-validation score before: 0.16666666666666666
Meta-validation score after:  1.0
Meta search candidate evaluations: 1


## Verify the changed improver on fresh learning episodes

These episodes are not used to select the policy. Compare old and new policies
using identical starting agents, adaptation examples, and candidate budgets.
The oracle remains hidden behind an evaluation function; its source is visible
here for education, so this toy is not a security boundary.

The table's budget counts candidate evaluations. One bundled candidate changes
three fields, so this is **not** a claim of less total compute or equal edit
complexity. Meta-search also costs extra and only pays off if reused enough.


In [6]:
meta_held_out_episodes = [
    ([invoice("6.25", quantity=6), invoice("80", discount="0.40"),
      invoice("19", shipping="7.25")],
     [invoice("22", quantity=3), invoice("35", discount="0.20"),
      invoice("29", shipping="1.50"),
      invoice("16", quantity=4, discount="0.25", shipping="8")]),
    ([invoice("30", quantity=2, discount="0.30", shipping="4")],
     [invoice("23", quantity=7, discount="0.10", shipping="2")]),
]
old_meta_score = evaluate_policy(initial_policy, meta_held_out_episodes).score
new_meta_score = evaluate_policy(reloaded_policy, meta_held_out_episodes).score

new_agent, new_log = improve(initial_agent, agent_evaluator, reloaded_policy, budget=1)
print(f"{'System':<28} {'Development':>12} {'Held-out':>12} {'Candidates':>12}")
for name, rules, count in [
    ("Initial agent", initial_agent, 0),
    ("Agent, old improver", reloaded_agent, 1),
    ("Agent, learned improver", new_agent, new_log["candidate_evaluations"]),
]:
    print(f"{name:<28} {accuracy(rules, development_tasks):>12.0%} "
          f"{accuracy(rules, held_out_tasks):>12.0%} {count:>12}")
print("Fresh episode average with old improver:", old_meta_score)
print("Fresh episode average with learned improver:", new_meta_score)

# Reuse the changed policy to run the same meta-improvement procedure again.
# This finite toy has reached its ceiling, so there is no further accepted edit.
policy_next, next_meta_log = improve(reloaded_policy, meta_evaluator,
                                   policy=reloaded_policy, budget=1)
print("Next recursive round:", asdict(policy_next),
      "; candidate evaluations:", next_meta_log["candidate_evaluations"])
assert accuracy(new_agent, held_out_tasks) == 1.0
assert new_meta_score == 1.0 and new_meta_score > old_meta_score
assert new_log["candidate_evaluations"] == agent_log["candidate_evaluations"] == 1
assert policy_next == reloaded_policy


System                        Development     Held-out   Candidates
Initial agent                          0%           0%            0
Agent, old improver                   33%          25%            1
Agent, learned improver              100%         100%            1
Fresh episode average with old improver: 0.125
Fresh episode average with learned improver: 1.0
Next recursive round: {'edit_mode': 'bundle'} ; candidate evaluations: 0


## Persistence audit and interpretation

| Level | Edited artifact | Persists? | Improvement procedure changed? |
|---|---|---|---|
| Output refinement | One invoice answer / temporary context | No | No |
| Agent self-improvement | `agent_rules.json` | Yes | No |
| Recursive self-improvement | `improvement_policy.json` | Yes | Yes: future proposals bundle repairs |

**What this proves:** a system whose executable code never changes can improve
its persistent task behavior and the configurable procedure that improves it.
The capacity to interpret editable data is sufficient in this bounded setting.

**What it does not prove:** autonomous discovery of a new search algorithm,
broad generalization, improvement of neural weights, or unlimited acceleration.
All candidate mechanisms exist in the initial interpreter. This is optimization
over a small human-designed space. Recursion reaches a ceiling after one change.

A code-writing agent could expand that space by adding new mechanisms. Coding
makes more of the system editable; it is not necessary for every kind of learning.
If a system exposes only source-code edits, then some component must write or
transform code. That component need not be the task-solving agent itself.

To turn this into an LLM experiment, replace deterministic patch diagnosis and
proposal generation with structured model output, retain the external evaluator,
and test on new task families. Never equate “a change was generated” with “the
change improved the system.”


In [7]:
audit = {
    "output_refinement": {"before": str(first_answer), "after": str(refined_answer),
                          "next_request": str(next_request_answer), "persistent_edit": False},
    "agent_self_improvement": {"rules": asdict(reloaded_agent),
                               "held_out_accuracy": accuracy(reloaded_agent, held_out_tasks)},
    "recursive_self_improvement": {"policy": asdict(reloaded_policy),
                                   "old_fresh_episode_score": old_meta_score,
                                   "new_fresh_episode_score": new_meta_score},
    "search_logs": {"agent": agent_log, "meta": meta_log, "new_agent": new_log},
}
(artifacts / "audit.json").write_text(json.dumps(audit, indent=2))
print("Saved inspectable artifacts:", ", ".join(sorted(p.name for p in artifacts.glob("*.json"))))
print("All demonstration assertions passed.")


Saved inspectable artifacts: agent_rules.json, audit.json, improvement_policy.json
All demonstration assertions passed.
